# 14 — Global Moran's I และ Getis-Ord Gi*: อัตราป่วยระดับตำบล จังหวัดสระบุรี

**คำถามที่สองวิธีนี้ตอบ**

| วิธี | คำถาม | ผลลัพธ์ |
|---|---|---|
| **Global Moran's I** | ทั้งจังหวัด ตำบลที่อัตราป่วยใกล้เคียงกัน **อยู่ติดกัน** มากกว่าที่จะเกิดโดยบังเอิญหรือไม่ | ตัวเลขเดียวทั้งพื้นที่ (I, p) |
| **Getis-Ord Gi\*** | **ตรงไหน** เป็นกลุ่มตำบลอัตราสูง (hot spot) หรือกลุ่มอัตราต่ำ (cold spot) | ค่า z และ p ของทุกตำบล → แผนที่ |

Global Moran's I บอกว่า "มีการเกาะกลุ่มไหม" ส่วน Gi* บอกว่า "เกาะกลุ่มที่ไหน" จึงใช้คู่กัน

**สิ่งที่ notebook นี้สอน (รันทีละเซลล์จากบนลงล่าง)**

| ขั้น | เนื้อหา |
|---|---|
| 1 | โหลดขอบเขตตำบลและข้อมูลป่วยจาก Google Drive อัตโนมัติ |
| 2 | ตรวจคุณภาพข้อมูล: ค่าว่าง และประชากรที่เป็นไปไม่ได้ |
| 3 | คำนวณอัตราป่วย + Empirical Bayes smoothing (แก้ปัญหาตำบลประชากรน้อย) |
| 4 | สร้าง spatial weights (ใครเป็นเพื่อนบ้านของใคร) |
| 5 | **Global Moran's I** + Moran scatterplot + permutation test |
| 6 | **Getis-Ord Gi\*** + การปรับ p-value (FDR) + แผนที่ hot/cold spot |
| 7 | Sensitivity: เปลี่ยน weights / ตัวแปร แล้วผลยังเหมือนเดิมไหม |
| 8 | ตรวจสอบว่ารูปแบบที่เห็นเป็นเรื่องจริงหรือเป็นผลจากวิธีบันทึกข้อมูล |
| 9 | สรุปผล ประโยคเขียนรายงาน และบันทึกไฟล์ |

> **ข้อมูลที่ใช้:** `saraburi_tambon_rates.gpkg` ขอบเขตตำบล 92 ตำบล (GADM) พร้อมจำนวนผู้ป่วย 11 โรคจาก HDC (รวม 4 ปี) และประชากรรายตำบล

## ขั้นที่ 0 ติดตั้งแพ็กเกจ
- `libpysal` — สร้าง spatial weights
- `esda` — Moran's I, Gi*, Empirical Bayes
- `mapclassify` — แบ่งช่วงสีแผนที่

(`geopandas` มีใน Colab แล้ว)

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
import sys, subprocess
for pkg, mod in [("geopandas", "geopandas"), ("libpysal", "libpysal"), ("esda", "esda"), ("mapclassify", "mapclassify")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("พร้อมใช้งาน")

## ขั้นที่ 1 ตั้งค่าและโหลดข้อมูลอัตโนมัติ

ลำดับการหาไฟล์: ① ในเครื่อง → ② ดาวน์โหลดจาก Google Drive ด้วย File ID (ครั้งแรกกด *Allow* หนึ่งครั้ง) → ③ mount Drive แล้วค้นหาชื่อไฟล์

**เลือกตัวแปรที่จะวิเคราะห์** ที่ `DISEASES` ค่าเริ่มต้นคือ **โรคระบบทางเดินหายใจ 7 โรค** รวมกัน
(การรวมโรคทำให้จำนวนผู้ป่วยต่อตำบลมากขึ้น อัตราจึงเสถียรกว่าการวิเคราะห์ทีละโรค)

In [ ]:
# ===================== ตั้งค่า (แก้ได้) =====================
FILE_NAME = "saraburi_tambon_rates.gpkg"
FILE_ID   = "1TJGoSDDsFY-aX6f_c5xn7-IBlW8WTZzw"
DRIVE_SEARCH_DIRS = ["MyDrive/000_research", "MyDrive"]

RESPIRATORY = ["copd", "asthma", "pneumonia", "influenza", "acute_pharyngitis", "chronic_rhinitis", "bronchitis"]
ALL_11 = RESPIRATORY + ["ischemic_heart", "stroke", "conjunctivitis", "dermatitis"]
DISEASES   = RESPIRATORY      # เปลี่ยนเป็น ALL_11 หรือ ["copd"] ได้
OUTCOME_LABEL = "Respiratory morbidity (7 diseases)"
N_YEARS    = 4                # จำนวนปีที่รวมไว้ในไฟล์
PER        = 1000             # อัตราต่อประชากรกี่คน
POP_MAX_PLAUSIBLE = 100_000   # ประชากรตำบลที่สูงกว่านี้ถือว่าผิดปกติ (ตรวจในขั้นที่ 2)
PERMUTATIONS = 999
ALPHA      = 0.05
SEED       = 42
CRS_METRIC = 32647            # UTM zone 47N (หน่วยเมตร)
OUT_DIR    = "spatial_autocorrelation_results"
# ============================================================

import io, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import libpysal
import esda
from esda.smoothing import Empirical_Bayes
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _download_by_id(file_id, dest):
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    req = build("drive", "v3").files().get_media(fileId=file_id)
    with io.FileIO(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _, done = dl.next_chunk()
    return Path(dest)


def _search_drive(name):
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    for sub in DRIVE_SEARCH_DIRS:
        root = Path("/content/drive") / sub
        if root.exists():
            hits = sorted(root.rglob(name))
            if hits:
                return hits[0]
    return None


def find_file(name, file_id):
    for p in ([Path(REPO_DATA) / name] if USE_REPO_DATA else []) + [Path(name), Path("data") / name, Path("data/geo") / name, Path("/content") / name]:
        if p.exists():
            print(f"[1] พบในเครื่อง: {p}")
            return p
    if not _in_colab():
        raise FileNotFoundError(f"ไม่พบ {name} — วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook หรือรันใน Colab")
    try:
        p = _download_by_id(file_id, f"/content/{name}")
        print(f"[2] ดาวน์โหลดจาก Google Drive แล้ว: {name}")
        return p
    except Exception as e:
        print(f"[2] ดาวน์โหลดด้วย ID ไม่สำเร็จ ({type(e).__name__}) → ค้นหาใน Drive")
    p = _search_drive(name)
    if p:
        print(f"[3] พบใน Drive: {p}")
        return p
    raise FileNotFoundError(f"หา {name} ไม่เจอทั้งใน Drive และในเครื่อง")


raw = gpd.read_file(find_file(FILE_NAME, FILE_ID)).to_crs(CRS_METRIC)
raw = raw.rename(columns={"NAME_2": "amphoe", "NAME_3": "tambon"})
raw["tambon"] = raw["tambon"].replace("NA", np.nan).fillna("(no name) " + raw["a_code"].astype(str))
print(raw.shape, raw.crs)
raw[["a_code", "amphoe", "tambon", "population"] + DISEASES[:3]].head()

## ขั้นที่ 2 ตรวจคุณภาพข้อมูล

Moran's I และ Gi* ไวต่อข้อมูลผิดพลาดมาก เพราะค่าผิดของตำบลเดียวจะส่งผลไปถึงตำบลเพื่อนบ้านทั้งหมด จึงตรวจ 2 เรื่องก่อน

1. **ตำบลที่ไม่มีข้อมูลผู้ป่วย** — คำนวณอัตราไม่ได้ ต้องตัดออก
2. **ประชากรที่เป็นไปไม่ได้** — ตัวหารผิดทำให้อัตราผิดทั้งหมด ตำบลในสระบุรีส่วนใหญ่มีประชากรไม่เกิน 10,000 คน ค่าที่สูงเกิน `POP_MAX_PLAUSIBLE` น่าจะเกิดจากการจับคู่รหัสตำบลผิด (เช่น รวมประชากรเทศบาลหลายแห่งเข้ามา)

> ตำบลที่ถูกตัดยังคงอยู่บนแผนที่ (สีเทา) แต่ไม่ถูกนำไปคำนวณ และ **spatial weights จะสร้างจากตำบลที่เหลือเท่านั้น**

In [ ]:
gdf = raw.copy()
gdf["cases"] = gdf[DISEASES].sum(axis=1, min_count=len(DISEASES))
gdf["person_years"] = gdf["population"] * N_YEARS

reason = pd.Series("", index=gdf.index)
reason[gdf["cases"].isna()] = "no case data"
reason[gdf["population"] > POP_MAX_PLAUSIBLE] = "implausible population"
reason[gdf["population"].isna() | (gdf["population"] <= 0)] = "no population"
gdf["excluded_reason"] = reason
excluded = gdf[gdf["excluded_reason"] != ""]

print(f"ประชากรตำบล: median = {gdf.population.median():,.0f}, Q3 = {gdf.population.quantile(.75):,.0f}, "
      f"max = {gdf.population.max():,.0f}")
print(f"ตัดออก {len(excluded)} จาก {len(gdf)} ตำบล:")
display(excluded[["a_code", "amphoe", "tambon", "population", "cases", "excluded_reason"]])

d = gdf[gdf["excluded_reason"] == ""].reset_index(drop=True)
print(f"เหลือวิเคราะห์ {len(d)} ตำบล, ประชากรรวม {d.population.sum():,.0f} คน, ผู้ป่วยรวม {d.cases.sum():,.0f} ครั้ง")

## ขั้นที่ 3 อัตราป่วย และ Empirical Bayes smoothing

**3.1 อัตราดิบ (crude rate)**

$$\text{rate}_i = \frac{\text{จำนวนผู้ป่วย}_i}{\text{ประชากร}_i \times \text{จำนวนปี}} \times 1{,}000$$

**3.2 ปัญหาของตำบลประชากรน้อย**

ตำบลที่มีประชากร 900 คน ถ้ามีผู้ป่วยเพิ่มขึ้นแค่ 10 ครั้ง อัตราจะกระโดดมาก ส่วนตำบล 10,000 คนแทบไม่เปลี่ยน
อัตราของตำบลเล็กจึง "แกว่ง" มากโดยบังเอิญ และมักกลายเป็นค่าสูงสุด/ต่ำสุดของจังหวัด ทั้งที่ไม่ได้ต่างจริง

**3.3 Empirical Bayes (EB) smoothing**

ดึงอัตราของแต่ละตำบลเข้าหาค่าเฉลี่ยของจังหวัด **มากหรือน้อยตามขนาดประชากร**
- ตำบลประชากรมาก → อัตราเชื่อถือได้ → ปรับน้อย
- ตำบลประชากรน้อย → อัตราไม่เสถียร → ปรับมาก

**3.4 ทำไมใช้ log**

อัตราป่วยรายตำบลเบ้ขวามาก ค่าสุดโต่งไม่กี่ตำบลจะครอบงำ Moran's I และ Gi* การ log ทำให้การกระจายสมมาตรขึ้น

In [ ]:
d["crude_rate"] = d["cases"] / d["person_years"] * PER
eb = Empirical_Bayes(d["cases"].values.reshape(-1, 1).astype(float),
                     d["person_years"].values.reshape(-1, 1).astype(float))
d["eb_rate"] = eb.r.flatten() * PER
d["log_eb_rate"] = np.log(d["eb_rate"])
d["log_crude_rate"] = np.log(d["crude_rate"].clip(lower=d.loc[d.crude_rate > 0, "crude_rate"].min() / 2))

summ = d[["crude_rate", "eb_rate", "log_eb_rate"]].describe().T
summ["skewness"] = [stats.skew(d[c]) for c in summ.index]
display(summ.round(2))
print(f"อัตราดิบต่างกันสูงสุด/ต่ำสุด {d.crude_rate.max() / d.loc[d.crude_rate > 0, 'crude_rate'].min():,.0f} เท่า")
shift = (np.abs(d.eb_rate - d.crude_rate) / d.crude_rate.clip(lower=1e-9) * 100).median()
print(f"EB ปรับอัตราไปเฉลี่ย (median) {shift:.2f}% — "
      + ("น้อยมาก เพราะความต่างระหว่างตำบลสูงกว่าความแกว่งจากขนาดประชากรมาก EB จึงแทบไม่ปรับ"
         if shift < 1 else "EB ช่วยลดความแกว่งของตำบลประชากรน้อยได้ชัดเจน"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, col, title in zip(axes, ["crude_rate", "eb_rate", "log_eb_rate"],
                          ["Crude rate per 1,000/yr", "EB-smoothed rate per 1,000/yr", "log(EB rate) — analysed"]):
    excluded.plot(ax=ax, color="lightgray", edgecolor="white", lw=.4)
    d.plot(column=col, ax=ax, cmap="YlOrRd", scheme="Quantiles", k=5, edgecolor="white", lw=.4,
           legend=True, legend_kwds={"fontsize": 7, "loc": "lower left", "fmt": "{:.1f}"})
    ax.set_title(title, fontsize=10); ax.set_axis_off()
fig.suptitle(f"{OUTCOME_LABEL}: gray = excluded", fontsize=11)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(d.population, d.crude_rate, s=18, alpha=.6, label="crude")
ax.scatter(d.population, d.eb_rate, s=18, alpha=.6, label="EB")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("population (log)"); ax.set_ylabel("rate per 1,000/yr (log)")
ax.set_title("Small tambons have the most extreme crude rates"); ax.legend()
plt.tight_layout(); plt.show()

## ขั้นที่ 4 Spatial weights: ใครเป็นเพื่อนบ้านของใคร

ทั้ง Moran's I และ Gi* ต้องรู้ว่าตำบลใดอยู่ติดกัน ข้อมูลนี้เก็บในเมทริกซ์ **W** ขนาด n × n

| แบบ | นิยามเพื่อนบ้าน | ใช้เมื่อ |
|---|---|---|
| **Queen contiguity** (ค่าหลัก) | มีขอบหรือมุมแตะกันแม้จุดเดียว | ขอบเขตการปกครองที่ติดกันเต็มพื้นที่ — นิยมที่สุด |
| Rook contiguity | ต้องมีขอบร่วมกัน (มุมไม่นับ) | เข้มงวดกว่า |
| K-nearest neighbors (KNN) | k ตำบลที่จุดศูนย์กลางใกล้ที่สุด | ทุกตำบลมีเพื่อนบ้านเท่ากัน ใช้ทดสอบความทนทานของผล |

**การปรับน้ำหนัก**
- **Row-standardized (`r`)** สำหรับ Moran's I: แต่ละแถวรวมเป็น 1 → ค่าเพื่อนบ้าน = ค่าเฉลี่ยของเพื่อนบ้าน
- **Binary (`B`)** สำหรับ Gi*: เพื่อนบ้าน = 1, ไม่ใช่ = 0 (ตามสูตรดั้งเดิมของ Getis-Ord)

> **Island** = ตำบลที่ไม่มีเพื่อนบ้านเลย ต้องตรวจเสมอ เพราะจะทำให้การคำนวณผิด

In [ ]:
w_queen = libpysal.weights.Queen.from_dataframe(d, use_index=False)
print(f"ตำบล {w_queen.n} | เพื่อนบ้านเฉลี่ย {w_queen.mean_neighbors:.1f} | "
      f"น้อยสุด {w_queen.min_neighbors} | มากสุด {w_queen.max_neighbors} | islands: {w_queen.islands or 'ไม่มี'}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
d.plot(ax=axes[0], color="#eee", edgecolor="gray", lw=.5)
cent = d.geometry.representative_point()
for i, nbrs in w_queen.neighbors.items():
    for j in nbrs:
        if j > i:
            axes[0].plot([cent[i].x, cent[j].x], [cent[i].y, cent[j].y], color="#c0504d", lw=.7)
cent.plot(ax=axes[0], color="k", markersize=5)
axes[0].set_title("Queen contiguity network"); axes[0].set_axis_off()
axes[1].hist(list(w_queen.cardinalities.values()), bins=range(0, w_queen.max_neighbors + 2), color="#4a7fb5", ec="white")
axes[1].set_xlabel("number of neighbors"); axes[1].set_ylabel("tambons"); axes[1].set_title("Neighbor count distribution")
plt.tight_layout(); plt.show()

## ขั้นที่ 5 Global Moran's I

**5.1 แนวคิด**

นำค่าของแต่ละตำบล (เทียบกับค่าเฉลี่ย) ไปคูณกับค่าเฉลี่ยของเพื่อนบ้าน (เทียบกับค่าเฉลี่ย)
- ตำบลสูงติดกับตำบลสูง หรือต่ำติดกับต่ำ → ผลคูณเป็นบวก → **I เป็นบวก** (เกาะกลุ่ม)
- ตำบลสูงติดกับตำบลต่ำ → ผลคูณเป็นลบ → **I เป็นลบ** (กระจายแบบสลับ เหมือนกระดานหมากรุก)

$$I = \frac{n}{\sum_i\sum_j w_{ij}} \cdot \frac{\sum_i\sum_j w_{ij}(x_i-\bar{x})(x_j-\bar{x})}{\sum_i (x_i-\bar{x})^2}$$

**ค่าคาดหวังเมื่อไม่มีรูปแบบเชิงพื้นที่** $E[I] = -1/(n-1)$ ซึ่งใกล้ 0 ไม่ใช่ 0 พอดี

**5.2 Permutation test (การทดสอบโดยสลับตำแหน่ง)**

สุ่มสลับค่าอัตราป่วยไปไว้ที่ตำบลต่างๆ 999 รอบ แล้วคำนวณ I ทุกรอบ ได้การกระจายของ I "ถ้าตำแหน่งไม่มีผล"
p-value = สัดส่วนของรอบที่ได้ I สุดโต่งเท่ากับหรือมากกว่าค่าจริง วิธีนี้ไม่ต้องสมมติว่าข้อมูลกระจายแบบปกติ

In [ ]:
w_queen.transform = "r"
y = d["log_eb_rate"].values
moran = esda.Moran(y, w_queen, permutations=PERMUTATIONS)
moran_eb = esda.Moran_Rate(d["cases"].values.astype(float), d["person_years"].values.astype(float),
                           w_queen, permutations=PERMUTATIONS)

def interp_I(I, EI, p):
    if p >= ALPHA:
        return "ไม่พบการเกาะกลุ่มเชิงพื้นที่ (กระจายแบบสุ่ม)"
    return "เกาะกลุ่ม (ค่าใกล้เคียงกันอยู่ติดกัน)" if I > EI else "กระจายแบบสลับ (ค่าสูงติดกับค่าต่ำ)"

global_res = pd.DataFrame([
    {"variable": "log(EB rate)", "I": moran.I, "E[I]": moran.EI, "z_sim": moran.z_sim, "p_sim": moran.p_sim,
     "interpretation": interp_I(moran.I, moran.EI, moran.p_sim)},
    {"variable": "EB Moran's I (Assunção-Reis)", "I": moran_eb.I, "E[I]": moran_eb.EI, "z_sim": moran_eb.z_sim,
     "p_sim": moran_eb.p_sim, "interpretation": interp_I(moran_eb.I, moran_eb.EI, moran_eb.p_sim)},
])
global_res.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
z = (y - y.mean()) / y.std()
lag = libpysal.weights.lag_spatial(w_queen, z)
ax = axes[0]
ax.scatter(z, lag, s=22, alpha=.7, color="#4a7fb5")
b = np.polyfit(z, lag, 1)
xs = np.linspace(z.min(), z.max(), 50)
ax.plot(xs, np.polyval(b, xs), color="crimson", lw=2, label=f"slope = Moran's I = {b[0]:.3f}")
ax.axhline(0, c="gray", lw=.8); ax.axvline(0, c="gray", lw=.8)
for (tx, ty, lab) in [(.97, .97, "High-High"), (.03, .03, "Low-Low"), (.03, .97, "Low-High"), (.97, .03, "High-Low")]:
    ax.text(tx, ty, lab, transform=ax.transAxes, ha="right" if tx > .5 else "left",
            va="top" if ty > .5 else "bottom", fontsize=9, color="gray")
ax.set_xlabel("standardized log(EB rate) of tambon"); ax.set_ylabel("spatial lag (mean of neighbors)")
ax.set_title("Moran scatterplot"); ax.legend(fontsize=8, loc="lower center")

ax = axes[1]
ax.hist(moran.sim, bins=40, color="lightgray", ec="white", label=f"{PERMUTATIONS} random permutations")
ax.axvline(moran.EI, color="k", ls="--", lw=1, label=f"E[I] = {moran.EI:.3f}")
ax.axvline(moran.I, color="crimson", lw=2.5, label=f"observed I = {moran.I:.3f}  (p = {moran.p_sim:.3f})")
ax.set_title("Permutation test"); ax.set_xlabel("Moran's I"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**วิธีอ่าน Moran scatterplot**
- แกน x = ค่าของตำบล, แกน y = ค่าเฉลี่ยของเพื่อนบ้าน (ทั้งคู่ปรับเป็น z-score)
- **ความชันของเส้นตรง = Moran's I** พอดี
- จุดส่วนใหญ่อยู่ช่อง High-High และ Low-Low → เกาะกลุ่ม; อยู่ช่อง High-Low และ Low-High → สลับ; กระจายทุกช่องเท่าๆ กัน → สุ่ม

**วิธีอ่าน permutation test**
- ถ้าเส้นสีแดง (ค่าจริง) อยู่กลางฮิสโตแกรม → ค่าที่ได้เกิดขึ้นได้โดยบังเอิญ
- ถ้าอยู่ไกลออกไปที่หาง → มีรูปแบบเชิงพื้นที่จริง

## ขั้นที่ 6 Getis-Ord Gi*

**6.1 แนวคิด**

สำหรับแต่ละตำบล ดูผลรวมของ **ตัวเองกับเพื่อนบ้านทั้งหมด** (เครื่องหมาย * หมายถึงรวมตัวเองด้วย) แล้วเทียบกับที่คาดไว้ถ้าค่ากระจายแบบสุ่ม

$$G_i^* = \frac{\sum_j w_{ij}x_j - \bar{X}\sum_j w_{ij}}{S\sqrt{\dfrac{n\sum_j w_{ij}^2 - \left(\sum_j w_{ij}\right)^2}{n-1}}}$$

ค่าที่ได้เป็น **z-score**
- z บวกมาก → **hot spot**: ตำบลนี้และเพื่อนบ้านมีค่าสูงพร้อมกัน
- z ลบมาก → **cold spot**: ตำบลนี้และเพื่อนบ้านมีค่าต่ำพร้อมกัน

> **สำคัญ:** ตำบลที่สูงมาก **ตำบลเดียว** แต่ล้อมรอบด้วยตำบลต่ำ **จะไม่เป็น hot spot** เพราะ Gi* ดูทั้งกลุ่ม ไม่ใช่ตำบลเดี่ยว

**6.2 ระดับความเชื่อมั่น** (ตามแบบ ArcGIS Hot Spot Analysis)

| ประเภท | เงื่อนไข |
|---|---|
| Hot spot 99% / 95% / 90% | z > 0 และ p < 0.01 / 0.05 / 0.10 |
| Cold spot 99% / 95% / 90% | z < 0 และ p < 0.01 / 0.05 / 0.10 |
| Not significant | อื่นๆ |

**6.3 ปัญหาการทดสอบหลายครั้ง**

ทดสอบ 88 ตำบลพร้อมกันที่ α = 0.05 จะได้ "hot/cold spot" ปลอมประมาณ 4 ตำบลแม้ข้อมูลเป็นแบบสุ่มทั้งหมด
จึงปรับด้วย **False Discovery Rate (FDR; Benjamini-Hochberg)** และรายงานทั้งก่อนและหลังปรับ

In [ ]:
w_bin = libpysal.weights.Queen.from_dataframe(d, use_index=False)
w_bin.transform = "B"
gi = esda.G_Local(y, w_bin, transform="B", star=True, permutations=PERMUTATIONS, seed=SEED)

def classify(zv, pv):
    out = []
    for zz, pp in zip(zv, pv):
        if pp < .01:   lvl = "99%"
        elif pp < .05: lvl = "95%"
        elif pp < .10: lvl = "90%"
        else:
            out.append("Not significant"); continue
        out.append(f"{'Hot' if zz > 0 else 'Cold'} spot {lvl}")
    return out

d["gi_z"] = gi.Zs
d["gi_p"] = gi.p_sim
d["gi_class"] = classify(d.gi_z, d.gi_p)
fdr_cut = esda.fdr(gi.p_sim, ALPHA)
d["gi_sig_fdr"] = d.gi_p <= fdr_cut

order = ["Hot spot 99%", "Hot spot 95%", "Hot spot 90%", "Not significant", "Cold spot 90%", "Cold spot 95%", "Cold spot 99%"]
print(d["gi_class"].value_counts().reindex(order, fill_value=0).to_string())
print(f"\nเกณฑ์ FDR (α = {ALPHA}): p ≤ {fdr_cut:.4f} → มีนัยสำคัญหลังปรับ {int(d.gi_sig_fdr.sum())} ตำบล")
print(f"ถ้าข้อมูลสุ่มทั้งหมด คาดว่าจะได้ p < 0.05 โดยบังเอิญ ≈ {len(d) * .05:.1f} ตำบล "
      f"(ได้จริง {int((d.gi_p < .05).sum())} ตำบล)")
sig = d[d.gi_p < .10].sort_values("gi_z", ascending=False)
sig[["tambon", "amphoe", "population", "cases", "crude_rate", "eb_rate", "gi_z", "gi_p", "gi_class", "gi_sig_fdr"]].round(3)

In [ ]:
COLORS = {"Hot spot 99%": "#b2182b", "Hot spot 95%": "#ef8a62", "Hot spot 90%": "#fddbc7", "Not significant": "#f7f7f7",
          "Cold spot 90%": "#d1e5f0", "Cold spot 95%": "#67a9cf", "Cold spot 99%": "#2166ac"}
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
ax = axes[0]
excluded.plot(ax=ax, color="lightgray", edgecolor="white", hatch="///", lw=.4)
d.plot(ax=ax, color=d["gi_class"].map(COLORS), edgecolor="gray", lw=.4)
for _, r in d[d.gi_p < .10].iterrows():
    pt = r.geometry.representative_point()
    ax.annotate(r.tambon, (pt.x, pt.y), fontsize=7, ha="center", fontweight="bold" if r.gi_sig_fdr else "normal")
ax.legend(handles=[Patch(fc=COLORS[k], ec="gray", label=k) for k in order] +
          [Patch(fc="lightgray", hatch="///", label="excluded")], fontsize=7, loc="lower left")
ax.set_title(f"Getis-Ord Gi* hot/cold spots — {OUTCOME_LABEL}\n(permutation p, unadjusted; bold = significant after FDR)", fontsize=10)
ax.set_axis_off()

ax = axes[1]
excluded.plot(ax=ax, color="lightgray", edgecolor="white", lw=.4)
lim = np.abs(d.gi_z).max()
d.plot(column="gi_z", ax=ax, cmap="RdBu_r", vmin=-lim, vmax=lim, edgecolor="gray", lw=.4, legend=True,
       legend_kwds={"shrink": .6, "label": "Gi* z-score"})
ax.set_title("Gi* z-score (continuous)", fontsize=10); ax.set_axis_off()
plt.tight_layout(); plt.show()

## ขั้นที่ 7 Sensitivity analysis

ผลเชิงพื้นที่ขึ้นกับการตัดสินใจหลายอย่าง ถ้าเปลี่ยนแล้วผลเปลี่ยนมาก แสดงว่าข้อสรุปยังไม่มั่นคง

| การทดสอบ | เปลี่ยนอะไร |
|---|---|
| A | Queen → KNN (k = 4, 6, 8) |
| B | log(EB rate) → log(crude rate) (ไม่ smooth) |
| C | ไม่ log (ใช้ EB rate ตรงๆ) |

In [ ]:
def run_moran(values, w):
    w.transform = "r"
    m = esda.Moran(values, w, permutations=PERMUTATIONS)
    return m.I, m.p_sim

def run_gi(values, w):
    w.transform = "B"
    g = esda.G_Local(values, w, transform="B", star=True, permutations=PERMUTATIONS, seed=SEED)
    return int(((g.Zs > 0) & (g.p_sim < .05)).sum()), int(((g.Zs < 0) & (g.p_sim < .05)).sum()), \
           int((g.p_sim <= esda.fdr(g.p_sim, ALPHA)).sum())

pts = d.copy(); pts["geometry"] = d.geometry.centroid
weights = {"Queen": lambda: libpysal.weights.Queen.from_dataframe(d, use_index=False)}
for k in (4, 6, 8):
    weights[f"KNN k={k}"] = (lambda k=k: libpysal.weights.KNN.from_dataframe(pts, k=k, use_index=False))
variables = {"log(EB rate)": d.log_eb_rate.values, "log(crude rate)": d.log_crude_rate.values, "EB rate (no log)": d.eb_rate.values}

rows = []
for vname, vals in variables.items():
    for wname, wf in weights.items():
        if vname != "log(EB rate)" and wname != "Queen":
            continue
        I, p = run_moran(vals, wf())
        hot, cold, n_fdr = run_gi(vals, wf())
        rows.append({"variable": vname, "weights": wname, "Moran_I": round(I, 3), "p_sim": p,
                     "Gi_hot_p<.05": hot, "Gi_cold_p<.05": cold, "Gi_sig_after_FDR": n_fdr})
sens = pd.DataFrame(rows)
sens

## ขั้นที่ 8 ตรวจสอบ: รูปแบบที่เห็นเป็นเรื่องจริงหรือเป็นผลจากวิธีบันทึกข้อมูล

**ประเด็นสำคัญของข้อมูล HDC:** ถ้าจำนวนผู้ป่วยถูกบันทึกตาม **ที่ตั้งสถานพยาบาล** แทน **ที่อยู่ของผู้ป่วย**
ตำบลที่มีโรงพยาบาลจะมีอัตราสูงผิดปกติ ส่วนตำบลรอบๆ จะต่ำ ผลที่ได้คือ "ยอดแหลม" กระจายเป็นจุดๆ ซึ่งทำให้:
- Global Moran's I ใกล้ 0 หรือติดลบ (สูงติดกับต่ำ)
- Gi* หา hot spot ไม่เจอ เพราะไม่มีกลุ่มตำบลที่สูงพร้อมกัน

เซลล์ด้านล่างตรวจ 3 อย่าง
1. อัตราดิบต่างกันกว่ากี่เท่า (ความต่างที่เป็นไปได้ทางระบาดวิทยามักไม่เกิน 5–10 เท่า)
2. ตำบลที่อัตราสูงสุด 15 อันดับแรกเป็นตำบลอะไร — **ให้ตรวจด้วยความรู้พื้นที่ว่าตรงกับที่ตั้งโรงพยาบาลหรือไม่**
3. ตำบลที่ชื่อเดียวกับอำเภอ (มักเป็นที่ตั้งตัวอำเภอและโรงพยาบาลชุมชน) มีอัตราสูงกว่าตำบลอื่นหรือไม่

In [ ]:
nz = d.loc[d.crude_rate > 0, "crude_rate"]
print(f"1) อัตราดิบ: ต่ำสุด {nz.min():.2f}, สูงสุด {nz.max():.1f} ต่อ 1,000/ปี → ต่างกัน {nz.max() / nz.min():,.0f} เท่า")
print(f"   ตำบลที่มีผู้ป่วยมากกว่าจำนวนประชากรในช่วง {N_YEARS} ปี: {int((d.cases > d.population).sum())} ตำบล "
      "(เป็นไปได้เฉพาะเมื่อนับเป็นครั้งที่มารับบริการ หรือรวมผู้ป่วยจากนอกตำบล)")

d["seat_tambon"] = [t.lower() in a.lower() for t, a in zip(d.tambon, d.amphoe)]
print("\n2) 15 ตำบลที่อัตราดิบสูงสุด:")
display(d.nlargest(15, "crude_rate")[["tambon", "amphoe", "population", "cases", "crude_rate", "seat_tambon", "gi_class"]].round(1))

seat, other = d.loc[d.seat_tambon, "crude_rate"], d.loc[~d.seat_tambon, "crude_rate"]
U, p = stats.mannwhitneyu(seat, other, alternative="greater")
print(f"3) ตำบลชื่อเดียวกับอำเภอ ({len(seat)} ตำบล): median = {seat.median():.1f}  vs  ตำบลอื่น ({len(other)}): "
      f"median = {other.median():.1f} ต่อ 1,000/ปี → {seat.median() / other.median():.0f} เท่า (Mann-Whitney p = {p:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7))
excluded.plot(ax=ax, color="lightgray", edgecolor="white", lw=.4)
d.plot(column="crude_rate", ax=ax, cmap="YlOrRd", scheme="Quantiles", k=5, edgecolor="white", lw=.4, legend=True,
       legend_kwds={"fontsize": 7, "loc": "lower left", "fmt": "{:.1f}", "title": "crude rate"})
top = d.nlargest(15, "crude_rate")
top.representative_point().plot(ax=ax, color="k", markersize=14)
for _, r in top.iterrows():
    pt = r.geometry.representative_point()
    ax.annotate(r.tambon, (pt.x, pt.y), xytext=(3, 3), textcoords="offset points", fontsize=7,
                fontweight="bold" if r.seat_tambon else "normal")
ax.set_title("Top-15 crude-rate tambons (bold = same name as district)\nIsolated spikes, not contiguous clusters", fontsize=10)
ax.set_axis_off(); plt.tight_layout(); plt.show()

## ขั้นที่ 9 สรุปผล ประโยคสำหรับเขียนรายงาน และบันทึกไฟล์

In [ ]:
def fmt_p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

n_hot = int(((d.gi_z > 0) & (d.gi_p < .05)).sum())
n_cold = int(((d.gi_z < 0) & (d.gi_p < .05)).sum())
n_fdr = int(d.gi_sig_fdr.sum())
moran_sig = moran.p_sim < ALPHA
report = (
    f"การวิเคราะห์สหสัมพันธ์เชิงพื้นที่ของอัตราป่วย{'' if DISEASES == RESPIRATORY else 'กลุ่มโรคที่เลือก'}"
    f"{'ด้วยโรคระบบทางเดินหายใจ 7 โรค' if DISEASES == RESPIRATORY else ''} ระดับตำบล ({len(d)} ตำบล; "
    f"ตัดออก {len(excluded)} ตำบลเนื่องจากข้อมูลไม่ครบหรือประชากรผิดปกติ) "
    f"โดยใช้ log ของอัตราที่ปรับด้วย Empirical Bayes และ Queen contiguity weights "
    f"พบว่า Global Moran's I = {moran.I:.3f} (E[I] = {moran.EI:.3f}, {fmt_p(moran.p_sim)}, {PERMUTATIONS} permutations) "
    f"{'แสดงว่ามีการเกาะกลุ่มเชิงพื้นที่อย่างมีนัยสำคัญ' if moran_sig and moran.I > moran.EI else 'แสดงว่าไม่พบการเกาะกลุ่มเชิงพื้นที่ที่มีนัยสำคัญ'} "
    f"และ EB Moran's I = {moran_eb.I:.3f} ({fmt_p(moran_eb.p_sim)}) "
    f"การวิเคราะห์ Getis-Ord Gi* พบ hot spot {n_hot} ตำบล และ cold spot {n_cold} ตำบล ที่ p < 0.05 "
    f"แต่{'ไม่มีตำบลใด' if n_fdr == 0 else f'มีเพียง {n_fdr} ตำบลที่'}มีนัยสำคัญหลังปรับด้วย FDR "
    + ("ผลดังกล่าวสอดคล้องกันในทุกรูปแบบของ spatial weights และตัวแปรที่ทดสอบ (sensitivity analysis) "
       if sens["p_sim"].lt(ALPHA).eq(moran_sig).all() and sens["Gi_sig_after_FDR"].eq(n_fdr).all()
       else "ทั้งนี้ผลเปลี่ยนไปเมื่อเปลี่ยน spatial weights หรือตัวแปร (ดูตาราง sensitivity) จึงควรตีความอย่างระมัดระวัง ")
    + f"นอกจากนี้อัตราป่วยดิบระหว่างตำบลต่างกันถึง {nz.max() / nz.min():,.0f} เท่า และตำบลที่ชื่อเดียวกับอำเภอมีค่า median สูงกว่าตำบลอื่น "
    f"{seat.median() / other.median():.0f} เท่า ซึ่งบ่งชี้ว่าข้อมูลอาจสะท้อนที่ตั้งสถานพยาบาลมากกว่าที่อยู่ของผู้ป่วย"
)
print(report)

In [ ]:
out = Path(OUT_DIR); out.mkdir(exist_ok=True)
keep = ["a_code", "amphoe", "tambon", "population", "cases", "crude_rate", "eb_rate", "log_eb_rate",
        "gi_z", "gi_p", "gi_class", "gi_sig_fdr", "seat_tambon", "geometry"]
d[keep].to_crs(4326).to_file(out / "tambon_gistar.gpkg", driver="GPKG")
d[keep[:-1]].to_csv(out / "tambon_gistar.csv", index=False)
global_res.to_csv(out / "global_moran.csv", index=False)
sens.to_csv(out / "sensitivity.csv", index=False)
excluded[["a_code", "amphoe", "tambon", "population", "cases", "excluded_reason"]].to_csv(out / "excluded_tambons.csv", index=False)
(out / "report_th.txt").write_text(report, encoding="utf-8")
if _in_colab() and Path("/content/drive/MyDrive").exists():
    import shutil
    shutil.copytree(out, Path("/content/drive/MyDrive") / OUT_DIR, dirs_exist_ok=True)
    print("คัดลอกผลไปที่ MyDrive/" + OUT_DIR)
print("บันทึกแล้ว:", sorted(p.name for p in out.iterdir()))
print("ไฟล์ .gpkg เปิดใน QGIS ได้ทันทีเพื่อทำแผนที่ฉบับตีพิมพ์")

## ข้อจำกัดและสิ่งที่ต้องทำต่อ

1. **ที่มาของจำนวนผู้ป่วย (สำคัญที่สุด)** — ต้องยืนยันว่า HDC นับตามที่อยู่ผู้ป่วยหรือตามสถานพยาบาล ถ้าเป็นตามสถานพยาบาล ผล Moran's I และ Gi* จะสะท้อนตำแหน่งโรงพยาบาล ไม่ใช่การกระจายของโรค ทางแก้คือขอข้อมูลที่จัดกลุ่มตามที่อยู่ของผู้ป่วย (ใน 43 แฟ้มใช้ข้อมูลที่อยู่และ TYPEAREA ของแฟ้ม PERSON) หรือวิเคราะห์ที่ระดับอำเภอแทน
2. **หน่วยนับเป็นครั้ง (visit) ไม่ใช่คน** — ผู้ป่วยโรคเรื้อรังหนึ่งคนถูกนับหลายครั้ง ตำบลที่มีผู้ป่วยเรื้อรังมากจึงมีค่าสูง ควรเรียกว่า "อัตราการมารับบริการ" หรือ morbidity rate ตามที่ตกลงไว้
3. **ขอบเขต GADM มี 92 ตำบล แต่ HDC มี 108 ตำบล** — บางตำบลถูกรวมหรือขาดหาย ควรใช้ขอบเขตตำบลของกรมการปกครองที่ตรงกับรหัส HDC ทั้ง 108 ตำบล
4. **ประชากร** — ตรวจตัวหารกับข้อมูลทะเบียนราษฎร์ หรือ WorldPop ก่อนใช้จริง
5. **Gi\* หาได้แค่กลุ่ม (cluster)** — ไม่สามารถตรวจตำบลที่สูงโดดเดี่ยว (spatial outlier) ถ้าต้องการหาทั้งกลุ่มและค่าโดด ให้ใช้ Local Moran's I (LISA) ซึ่งแบ่งได้เป็น High-High, Low-Low, High-Low, Low-High

**อ้างอิง**
- Moran, P. A. P. (1950). Notes on continuous stochastic phenomena. *Biometrika, 37*(1/2), 17–23.
- Getis, A., & Ord, J. K. (1992). The analysis of spatial association by use of distance statistics. *Geographical Analysis, 24*(3), 189–206.
- Ord, J. K., & Getis, A. (1995). Local spatial autocorrelation statistics: Distributional issues and an application. *Geographical Analysis, 27*(4), 286–306.
- Assunção, R. M., & Reis, E. A. (1999). A new proposal to adjust Moran's I for population density. *Statistics in Medicine, 18*(16), 2147–2162.
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate. *Journal of the Royal Statistical Society: Series B, 57*(1), 289–300.